# Understanding Large Language Models - Lab 1: Setting Up Your Ecosystem

## Introduction
### This notebook will guide you through setting up your environment for the course.
### We will install the necessary dependencies, download a pre-trained T5 model, and run a simple text-to-text prediction.

In [1]:
!pip install transformers torch sentencepiece datasets transformers[torch]

  Using cached sentencepiece-0.2.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached sympy-1.14.0-py3

In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

def generate_text(input_text, max_length=50):
    
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids

    ### PRINT OUT input_ids
    
    output_ids = model.generate(input_ids, max_length=max_length)

    ### PRINT OUT output_ids
    
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


example_input = "translate to french: I love learning new things."
output_text = generate_text(example_input)

print("Input:", example_input)
print("Output:", output_text)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 6577.76it/s]


Input: translate to french: I love learning new things.
Output: Je suis très enthousiaste à apprendre de nouvelles choses.


In [8]:
### Try at least 5 other example inputs
### Example 1: "How are you?"
### Example 2: "What is your name?"
### Example 3: "Where is the nearest restaurant?"
### Example 4: "I love learning new things."
### Example 5: "This is a beautiful day."

# T5 and the Prefix + Input Structure

T5 (Text-to-Text Transfer Transformer) is explicitly trained to follow a **prefix + input** format, guiding it to perform the correct NLP task.

## Why Prefixes Matter
T5 was trained using structured prompts like:
- `translate English to French: How are you?` → `Comment allez-vous?`
- `summarize: The Eiffel Tower is in Paris.` → `Eiffel Tower is in Paris.`
- `question: Who discovered gravity? context: Isaac Newton discovered gravity.` → `Isaac Newton`
- `sentiment: I love this movie!` → `positive`

## Without a Prefix?
❌ `How are you?` → (Unpredictable output)  
✅ `translate English to French: How are you?` → `Comment allez-vous?`

## Custom Prefixes
Fine-tune T5 with your own prefixes:
- `explain: What is...`
- `medical diagnosis: Patient has high fever...`

In [9]:
import torch
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset

csv_filename = "explain_dataset.csv"

df = pd.read_csv(csv_filename)
dataset = Dataset.from_pandas(df)

def preprocess_function(examples):
    inputs = examples["Input"]
    targets = examples["Response"]
    
    model_inputs = tokenizer(inputs, max_length=64, truncation=True, padding="max_length")
    
    labels = tokenizer(targets, max_length=64, truncation=True, padding="max_length").input_ids

    model_inputs["labels"] = labels
    
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

dataset_split = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

training_args = TrainingArguments(
    output_dir="./t5-fine-tuned",
    eval_strategy="epoch",      # Changed from evaluation_strategy
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_strategy="epoch",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

# Save the fine-tuned model
model.save_pretrained("./t5-custom-response")
tokenizer.save_pretrained("./t5-custom-response")

print("Fine-tuning complete! Model saved to ./t5-custom-response")

Map: 100%|██████████| 1000/1000 [00:00<00:00, 9972.10 examples/s]


Epoch,Training Loss,Validation Loss
1,No log,0.003302
2,No log,0.000385
3,0.235523,0.000242


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.60it/s]

Fine-tuning complete! Model saved to ./t5-custom-response


In [10]:
original_model = T5ForConditionalGeneration.from_pretrained("t5-small")
fine_tuned_model = T5ForConditionalGeneration.from_pretrained("./t5-custom-response")

def generate_response(model, input_text):
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids
    output_ids = model.generate(input_ids, max_length=64)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

test_question = "explain: What is machine learning?"

original_output = generate_response(original_model, test_question)
fine_tuned_output = generate_response(fine_tuned_model, test_question)

print("Original Model Output:")
print(original_output)
print("\n\n\nFine-Tuned Model Output:")
print(fine_tuned_output)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 14965.78it/s]


Original Model Output:
Warum ist die Frage, wie machmach learning?



Fine-Tuned Model Output:
Machine learning is a subset of AI that enables systems to learn from data and improve without explicit programming.
